# Annual Report Summary Examples

Choose **Generate a good summary** or **Create a corrupted example**. Good-summary generation keeps its existing prompts unchanged. Corruption mode lets you select one or several general error types and choose **Light** or **Heavy** severity. Corrupted examples are clearly labeled and saved separately from good summaries.

This project does not include a model provider or credentials. Before generating summaries or corrupted examples, implement the async callable in `student_llm_client.py` for a provider you are permitted to use.

In [ ]:
# 1. Environment Setup
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Markdown, display

print("Notebook imports ready")

In [ ]:
# 2. Project Paths and Run Settings
# Find this standalone project so notebook runs do not depend on the current directory.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").is_file()
        and (candidate / "summarize_annual_reports.ipynb").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the annual-report summary project root.")

INPUT_FOLDER = PROJECT_ROOT
REPORT_FOLDERS = {
    "business_description": INPUT_FOLDER / "business_reports",
    "sustainability": INPUT_FOLDER / "sustainability_reports",
}
OUTPUT_ROOT = INPUT_FOLDER / "generated_summaries"
OUTPUT_FOLDERS = {
    "business_description": OUTPUT_ROOT / "business",
    "sustainability": OUTPUT_ROOT / "sustainability",
}
CORRUPTED_OUTPUT_FOLDERS = {
    "business_description": INPUT_FOLDER / "corrupted_examples" / "business",
    "sustainability": INPUT_FOLDER / "corrupted_examples" / "sustainability",
}

MAX_COMPLETION_TOKENS = 4_096
MAX_CHUNK_CHARACTERS = 18_000
MAX_EVIDENCE_CHARACTERS = 24_000
MAX_CONSOLIDATION_CHARACTERS = 5_000
MAX_RETRIES = 3
RETRY_BACKOFF_SECONDS = 3

In [ ]:
# 3. Initialize the Summary Workflow and Student Model Adapter
import importlib
import inspect

import annual_report_summarizer
import student_llm_client
import summary_corruptor

# Reload local modules so code edits are picked up without restarting the kernel.
annual_report_summarizer = importlib.reload(annual_report_summarizer)
student_llm_client = importlib.reload(student_llm_client)
summary_corruptor = importlib.reload(summary_corruptor)
AnnualReportSummarizer = annual_report_summarizer.AnnualReportSummarizer
StudentLLMClient = student_llm_client.StudentLLMClient

# Close only the previous student adapter; do not depend on any legacy client object.
if "model_client" in globals():
    await model_client.close()
model_client = StudentLLMClient()
summarizer = AnnualReportSummarizer(
    model_client,
    max_chunk_characters=MAX_CHUNK_CHARACTERS,
    max_evidence_characters=MAX_EVIDENCE_CHARACTERS,
    max_consolidation_characters=MAX_CONSOLIDATION_CHARACTERS,
    max_completion_tokens=MAX_COMPLETION_TOKENS,
    max_retries=MAX_RETRIES,
    retry_backoff_seconds=RETRY_BACKOFF_SECONDS,
)
assert "summary_type" in inspect.signature(summarizer.run_reports).parameters
print(f"Workflow loaded: {annual_report_summarizer.__file__}")
print("Student model adapter is ready for implementation.")

In [ ]:
# 4. Choose Summary Type, Action, Corruption Types, Severity, and Run Mode
task_selector = widgets.Dropdown(
    options=[
        ("Generate a good summary", "good"),
        ("Create a corrupted example from a saved good summary", "corrupt"),
    ],
    value="good",
    description="Action:",
)
summary_type_selector = widgets.Dropdown(
    options=[
        ("Sustainability performance assessment", "sustainability"),
        ("Business description", "business_description"),
    ],
    value="sustainability",
    description="Summary type:",
)
corruption_type_selector = widgets.SelectMultiple(
    options=[("All corruption types", "__all__")],
    value=(),
    description="Corruption types:",
    layout=widgets.Layout(width="95%", height="145px"),
    disabled=True,
)
severity_selector = widgets.Dropdown(
    options=summary_corruptor.available_severities(),
    value="light",
    description="Severity:",
    disabled=True,
)
run_mode = widgets.Dropdown(
    options=["Single report", "Batch all reports"],
    value="Single report",
    description="Run mode:",
)
report_selector = widgets.Dropdown(
    options=[("Select a summary type", "")],
    value="",
    description="Report:",
    layout=widgets.Layout(width="95%"),
)
report_paths = []


def refresh_report_options(change=None):
    """Refresh report choices and corruption types for the selected summary type."""
    global report_paths
    summary_type = summary_type_selector.value
    if not isinstance(summary_type, str):
        raise ValueError("Select a summary type.")
    corruption_type_selector.options = [
        ("All corruption types", "__all__"),
        *summary_corruptor.available_general_corruption_types(),
    ]
    corruption_type_selector.value = ()
    report_paths = AnnualReportSummarizer.discover_reports(REPORT_FOLDERS[summary_type])
    if report_paths:
        default_report = next(
            (path for path in report_paths if path.name.startswith("vdl-jaarverslag")),
            report_paths[0],
        )
        report_selector.options = [(path.name, str(path)) for path in report_paths]
        report_selector.value = str(default_report)
        report_selector.disabled = run_mode.value == "Batch all reports"
        print(f"Found {len(report_paths)} PDF(s) in {REPORT_FOLDERS[summary_type]}.")
    else:
        report_selector.options = [(f"No PDFs found in {REPORT_FOLDERS[summary_type].name}", "")]
        report_selector.value = ""
        report_selector.disabled = True
        print(f"No PDFs found in {REPORT_FOLDERS[summary_type]} yet.")
    update_corruption_controls()


def update_report_selector(change):
    """Disable the individual report picker while batch mode is selected."""
    report_selector.disabled = change["new"] == "Batch all reports" or not report_paths


def update_corruption_controls(change=None):
    """Show corruption-only controls only when creating a bad example."""
    corrupt_mode = task_selector.value == "corrupt"
    corruption_type_selector.disabled = not corrupt_mode
    severity_selector.disabled = not corrupt_mode


summary_type_selector.observe(refresh_report_options, names="value")
task_selector.observe(update_corruption_controls, names="value")
run_mode.observe(update_report_selector, names="value")
display(
    widgets.VBox(
        [task_selector, summary_type_selector, corruption_type_selector, severity_selector, run_mode, report_selector]
    )
)
refresh_report_options()

In [ ]:
# 5. Generate a Good Summary or a Labeled Corrupted Example
mode = run_mode.value
action = task_selector.value
summary_type = summary_type_selector.value

# Validate the widget state before starting any report work or model requests.
if mode not in {"Single report", "Batch all reports"}:
    raise ValueError(f"Unsupported run mode: {mode!r}")
if summary_type not in REPORT_FOLDERS:
    raise ValueError(f"Unsupported summary type: {summary_type!r}")
if action not in {"good", "corrupt"}:
    raise ValueError(f"Unsupported action: {action!r}")
if not report_paths:
    raise FileNotFoundError(
        f"No PDFs found in {REPORT_FOLDERS[summary_type]}. "
        "Add reports to that folder or choose the other summary type."
    )

if mode == "Single report":
    report_value = report_selector.value
    if not isinstance(report_value, str) or not report_value:
        raise ValueError("Select a report before starting the run.")
    selected_paths = [Path(report_value)]
else:
    selected_paths = report_paths

if action == "good":
    # Good-summary prompts remain separate from the provider-specific model adapter.
    output_folder = OUTPUT_FOLDERS[summary_type]
    print(f"Generating {len(selected_paths)} good summary/summaries.")
    print(f"Source: {REPORT_FOLDERS[summary_type]}")
    print(f"Output: {output_folder}")
    run_results = await summarizer.run_reports(
        selected_paths,
        output_folder,
        summary_type=summary_type,
    )
    for result in run_results:
        display(Markdown(f"## {result['filename']}\n\n{result['summary']}"))
else:
    selected_corruptions = tuple(corruption_type_selector.value)
    if not selected_corruptions:
        raise ValueError("Select one or more corruption types, or choose All corruption types.")
    severity = severity_selector.value
    if severity not in {"light", "heavy"}:
        raise ValueError(f"Unsupported corruption severity: {severity!r}")

    output_folder = CORRUPTED_OUTPUT_FOLDERS[summary_type]
    corruption_results = []
    for report_path in selected_paths:
        try:
            good_summary_path = summary_corruptor.find_good_summary(
                report_path,
                OUTPUT_FOLDERS[summary_type],
                summary_type,
                legacy_output_folder=OUTPUT_ROOT,
            )
            corrupted_path, manifest = await summary_corruptor.save_general_corruption(
                good_summary_path,
                output_folder,
                summary_type,
                selected_corruptions,
                severity,
                source_filename=report_path.name,
                llm_client=summarizer.llm_client,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )
            corruption_results.append(
                {
                    "filename": report_path.name,
                    "path": corrupted_path,
                    "manifest": manifest,
                    "label": manifest["corruption_label"],
                    "severity": manifest["severity"],
                }
            )
            print(
                f"Created {severity} corruption for {report_path.name}: "
                f"{manifest['corruption_label']}"
            )
        except (FileNotFoundError, ValueError, KeyError) as error:
            print(f"Skipped {report_path.name}: {error}")

    # Manifests stay in memory; only marked Markdown examples are saved.
    for result in corruption_results:
        print(f"Markdown: {result['path']}")
        if mode == "Single report":
            display(Markdown(result["path"].read_text(encoding="utf-8")))
    print(f"Created {len(corruption_results)} corrupted example(s) in {output_folder}")